# 10 — Kaggle submission by registry `model_id`

**Summary:** Loads one or more adapters from `<WORKFLOW_ROOT>/models/model_registry.csv`, runs **raw** generation on the **official test CSV** (prompts only), applies **`POSTPROCESS_METHOD`** for submission safety, validates rows, and writes `submission.csv`.

**Parameters:** `RUN_PROFILE_ID` (match notebook **03**), `MODEL_IDS`, `TEST_CSV`, `SUBMISSION_PATH`, `POSTPROCESS_METHOD`, `max_new_tokens` (per model from registry `max_new_tokens_default` when present).

#### Colab / install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install transformers peft accelerate bitsandbytes pandas tqdm lxml

#### Parameters + run

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
import torch
from tqdm.auto import tqdm

PROJECT_DIR = Path('/content/drive/MyDrive/DL_Midterm_Spring_2026_2/svg_project_DL')
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

RAW_DIR = PROJECT_DIR / 'data' / 'raw'
RUN_PROFILE_ID = 'default'
WORKFLOW_ROOT = PROJECT_DIR / 'outputs' / 'workflow_runs' / RUN_PROFILE_ID
MODELS_ROOT = WORKFLOW_ROOT / 'models'
OUTPUTS_DIR = PROJECT_DIR / 'outputs'

MODEL_IDS = ['REPLACE_WITH_REGISTRY_MODEL_ID']
BASE_MODEL_ID = 'Qwen/Qwen2.5-Coder-1.5B-Instruct'
TEST_CSV = RAW_DIR / 'test.csv'
SUBMISSION_PATH = OUTPUTS_DIR / 'submissions' / 'submission.csv'
POSTPROCESS_METHOD = 'current_default_sanitizer'
MAX_NEW_TOKENS_FALLBACK = 768

from src.core.dataframe import choose_first_existing
from src.eval.postprocess_presets import get_postprocess_fn
from src.inference.generation import generate_svg_raw_prediction
from src.svg.cleaning import validate_svg_constraints
from src.training.lora.modeling import load_inference_adapter
from src.training.lora.registry import load_registry, resolve_adapter_path
from src.training.prompts import format_svg_instruction_example

test_df = pd.read_csv(TEST_CSV)
if 'id' not in test_df.columns:
    raise ValueError('test CSV needs id column')
prompt_col = choose_first_existing(test_df, ['prompt', 'description', 'text'], 'test_df')

reg = load_registry(PROJECT_DIR, models_root=MODELS_ROOT)
post_fn = get_postprocess_fn(POSTPROCESS_METHOD)
rows = []
for mid in MODEL_IDS:
    rrow = reg[reg['model_id'].astype(str) == str(mid)]
    if len(rrow) != 1:
        raise ValueError(f'model_id not unique in registry: {mid}')
    rrow = rrow.iloc[0]
    adapter = resolve_adapter_path(PROJECT_DIR, str(rrow['adapter_dir']))
    meta = json.loads(rrow['training_config_json'])
    mxt = int(meta.get('max_new_tokens_default', MAX_NEW_TOKENS_FALLBACK))
    tokenizer, model = load_inference_adapter(adapter, BASE_MODEL_ID)
    for _, trow in tqdm(test_df.iterrows(), total=len(test_df)):
        pid = trow['id']
        prompt_text = str(trow[prompt_col])
        full_prompt = format_svg_instruction_example(prompt_text, svg_text=None, include_answer=False)
        raw = generate_svg_raw_prediction(full_prompt, tokenizer, model, max_new_tokens=mxt)
        svg = post_fn(raw)
        rows.append({'id': pid, 'svg': svg, 'registry_model_id': mid})
    del model, tokenizer

# If multiple MODEL_IDS, last model wins per row; usually pass one id
sub = pd.DataFrame(rows)
if len(MODEL_IDS) > 1:
    sub = sub.drop_duplicates(subset=['id'], keep='last')
sub[['id', 'svg']].to_csv(SUBMISSION_PATH, index=False)
print('Wrote', SUBMISSION_PATH, sub.shape)